# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rebha-ds/flyrank-first-ml-assignment/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**CTR / Engagement Opportunity Scoring**

I'm currently aiming to be a product analyst and this track will help me grow in the path I chose. I also enjoy understanding how users interact with products, measuring outcomes through data.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Which visible pages under-capture clicks or engagement and deserve metadata, content, or monitoring review?**

My work will improve three decisions:
*   Whether to delete/redirect a page or leave it alone
*   a ranked list of CTR or engagement review candidates
*   reason codes such as high impressions, low CTR, strong position, enough volume, enough sessions, or weak engagement
*   action suggestions such as rewrite title/meta, improve intent match, improve
snippet structure, improve on-page engagement, or monitor.


---



The people who act over these decisions:   (Used Gemini to research this)
1.   SEO Specialists and Growth Marketers
2.   Content Strategy and Copywriter
3. Front end engineers and SREs


---


What does a wrong recommendation cost ?

Technical cost:
*  A high False Negative Rate (FNR) could lead to losing potential improvement in CTR / Engagement, eventually leading to less business impact.
*   A high False Positive Rate (FPR) could lead to wasting hours reviewing perfectly fine pages.

Business cost:
*   If the ML recommends changing a title tag to boost clicks, but the change causes the page to lose its organic search ranking entirely, the recommendation has a net-negative financial impact.(Used Gemini to research this)
*   If a rewrite unintentionally strips out core keywords that kept the page in its "Strong Position" to begin with, the recommendation can accidentally destroy the page's organic visibility entirely. (Used Gemini to research this)



**Narrowing the scope to match one model, one problem:**

Restating this cleanly: the decision my model actually supports is *which existing pages should get a human content/SEO review, and why* — a ranked list with reason codes and a suggested next step (rewrite title/meta, improve intent match, improve on-page engagement, or monitor).

Redirecting or deleting a page, and having the search/recommendation algorithm demote a page, are **not** outputs of this model. Those are separate, higher-stakes actions with their own risk profile that a human might take *after* reading a flagged page — not something a score tuned for "worth a review" should ever trigger directly. Keeping the model's job to one thing (rank + explain) is also what keeps the cost framing below honest: a wrong "review this" costs a few wasted minutes, while a wrong "redirect this" or "demote this" could cost real traffic — a different problem, needing a different validation bar, that I'm deliberately keeping out of this model's scope.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [44]:
import pandas as pd
data = pd.read_csv("data/raw/content_refresh_anonymized.csv")


# --- 1. Scale: how many pages does this actually touch? ---
data['is_underperformer'] = (data['ctr'] <= data['ctr'].median()).astype(int)
n_underperformers = data['is_underperformer'].sum()
print(f"1. SCALE: {n_underperformers:,} of {len(data):,} pages ({n_underperformers/len(data):.1%}) "
      f"fall below median CTR.")

# --- 2. Magnitude: how much worse are they? ---
eng_under = data.loc[data['is_underperformer'] == 1, 'engagement_rate'].mean()
eng_rest = data.loc[data['is_underperformer'] == 0, 'engagement_rate'].mean()
pct_gap = (eng_under - eng_rest) / eng_rest
print(f"2. MAGNITUDE: engagement rate is {eng_under:.2f} for underperformers vs {eng_rest:.2f} "
      f"for the rest ({pct_gap:.1%}).")

# --- 3. A concrete, actionable driver ---
by_content_type = data.groupby('content_type', observed=True)['is_underperformer'].mean().sort_values(ascending=False)
print("3. ACTIONABLE DRIVER: underperformer rate by content_type:")
print(by_content_type.round(3))

1. SCALE: 15,224 of 30,000 pages (50.7%) fall below median CTR.
2. MAGNITUDE: engagement rate is 1.45 for underperformers vs 3.65 for the rest (-60.3%).
3. ACTIONABLE DRIVER: underperformer rate by content_type:
content_type
comparison article    0.835
feedly article        0.738
keyword article       0.481
Name: is_underperformer, dtype: float64


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim:**
- *Observed*: in this export, the CTR-bucketed table shows engagement rising step by step as CTR rises — the lowest CTR decile has the lowest average engaged sessions, the top decile the highest, and this holds inside every `position_tier`, not just overall.
- *Observed*: the CTR<=7% underperformer cohort (about half the dataset) is measurably deeper-ranked on average (`avg_position` further down) and lower-engagement than the rest — but its scroll rate is actually *higher*, not lower, than the overall baseline. That's a real anomaly in the data, not something I have an explanation for yet, so I'm naming it rather than smoothing it over.
- *Directional*: the rule engine's four buckets (rewrite title/meta, improve intent match, improve on-page engagement, monitor) are a reasonable first sort of *where to look*, because they're built directly from the same CTR/engagement/scroll patterns above — not a claim that acting on any single row will move its numbers.
- *Decision-support*: the AI-model and intent breakdowns are useful context for the human reviewing a flagged page ("this page's model/intent mix is unusual for this cohort") — not a claim that `model_used` caused the page's CTR or engagement.

**What I can never claim:**
- Not causal. I have correlations from one export, not an experiment — I can't say a low CTR *causes* low engagement, or that a title/meta rewrite *will* raise CTR. Section 2 already names the real risk here: a rewrite could just as easily strip the keywords holding a "Strong Position" and make things worse, which is exactly why this stays a *recommendation*, not an automated action.
- Not "predicting Google." `avg_position` and `trend_direction` reflect this client's own observed search-console history, not a model of how the ranking algorithm actually works.
- Not a guarantee for any individual page. A page landing in the "Rewrite title/meta" bucket resembles other low-CTR, high-impression pages in this dataset — it doesn't mean *this* page is actually broken. The reason codes exist so a human can check that before anyone touches the page.

In [50]:
# Quick, honest gut-check behind the "directional, not causal" claim above:
# how strong is the CTR-engagement relationship, and does it hold within every position tier
# (not just because good positions happen to have both high CTR and high engagement)?

overall_corr = data['ctr'].corr(data['engagement_rate'])
print(f"Overall correlation, ctr vs engagement_rate: {overall_corr:.3f}")

within_tier_corr = (
    data.groupby('position_tier', observed=True)
    .apply(lambda g: g['ctr'].corr(g['engagement_rate']))
    .rename('ctr_engagement_corr')
)
print("\nSame correlation, computed separately within each position tier:")
print(within_tier_corr)

print(
    "\nThe relationship survives within-tier, so it isn't purely an artifact of "
    "'good positions get both good CTR and good engagement' \u2014 but a surviving correlation "
    "is still not causation, which is why the claims above stay observed/directional."
)

Overall correlation, ctr vs engagement_rate: 0.097

Same correlation, computed separately within each position tier:
position_tier
deep        0.050445
page_1      0.118434
page_3_5    0.056742
striking    0.085091
top_3       0.159004
Name: ctr_engagement_corr, dtype: float64

The relationship survives within-tier, so it isn't purely an artifact of 'good positions get both good CTR and good engagement' — but a surviving correlation is still not causation, which is why the claims above stay observed/directional.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.